In [19]:
import os
from utils_extraction import extract_body, tokenize, clean_tokens, decode, is_auto_label_tag, is_tag_token
from utils_extraction import chunk_tokens, flatten_token_chunks, merge_sentences_with_heuristics_tokens
from utils_extraction import extract_few_shot_examples
from utils_extraction import select_few_shot, prepare_label_tokens
from utils_extraction import merge_tokens_with_auto_labels, merge_tokens_general, add_attributes_to_auto_labels, compare_html_allow_auto_labels, correct_tokens_brackets, check_tokens_brackets
from models import GPTAssistant
from utils_extraction import process_chunks
from utils_extraction import clean_html_formatting


import json
import spacy
import re

In [2]:
project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"


In [9]:
def main_chunk_html(html_content, min_tokens=500, method="sentence"):

    if method == "sentence":
        return chunk_html_by_sentence(html_content, min_tokens=min_tokens)
    elif method == "paragraph":
        return chunk_html_by_paragraph(html_content, min_tokens=min_tokens)
    else:
        raise ValueError(f"Invalid method: {method}. Choose 'sentence' or 'paragraph'.")
    


def chunk_html_by_sentence(html_content, min_tokens=500):
    body_content = extract_body(html_content)


    # ---------- Tokenize body content ----------
    tokens = tokenize(body_content)

    # ---------- Clean tokens ----------
    normalized_cleaned_tokens = clean_tokens(html_tokens=tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

    

    nlp = spacy.load("en_core_web_trf")

    doc = nlp(decode(normalized_cleaned_tokens))
    initial_sentences = [sent.text for sent in doc.sents]

    initial_sentences_token = [tokenize(sent) for sent in initial_sentences]
    flat_initial_sentences = flatten_token_chunks(initial_sentences_token, separator="<sep>")


    is_sep_tag = lambda token: token == '<sep>'

    # Example: merge normalized_cleaned_tokens (original, no <sep>) 
    # with flat_token_sentence_chunks (derived, with <sep>)

    print(f"Original tokens: {len(normalized_cleaned_tokens)} (no <sep>)")
    print(f"Derived tokens: {len(flat_initial_sentences)} (with <sep>)")

    # This assumes flat_token_sentence_chunks is normalized_cleaned_tokens + <sep> insertions
    corrected_initial_sentences = merge_tokens_general(
        original_tokens=normalized_cleaned_tokens,
        derived_tokens=flat_initial_sentences,
        is_protected_func=is_sep_tag,
        log=False
    )

    print(f"\nResult: {len(corrected_initial_sentences)} tokens")
    print(f"Number of <sep> tags: {corrected_initial_sentences.count('<sep>')}")

    # Apply heuristic-based merging with sequential citation detection
    CITATION_THRESHOLD = 25  # Combined density threshold (%) - adjust based on the graphs above

    flat_token_sentence_chunks = merge_sentences_with_heuristics_tokens(corrected_initial_sentences, citation_threshold=CITATION_THRESHOLD, min_tokens=min_tokens)
    print(f"Initial sentences: {len(corrected_initial_sentences)}, After merging: {len(flat_token_sentence_chunks)}")
    print(f"Using citation threshold: {CITATION_THRESHOLD}% (combined period + number density)")
    print(f"Gap tolerance: 3 consecutive sentences below threshold to end citation section")
    print(f"Note: Only the FIRST citation section is detected; all subsequent sentences are not citations")


    token_chunks =  []
    current_chunk = []

    for token in flat_token_sentence_chunks:

        if token != "<sep>":
            current_chunk.append(token)
        else:
            token_chunks.append(current_chunk)
            current_chunk =  []
    token_chunks.append(current_chunk)

    # Verify: merged should equal normalized_cleaned_tokens if ignoring <sep> tag

    if flatten_token_chunks(token_chunks) == normalized_cleaned_tokens:
        print("✓ Perfect match! Derived was indeed original + <sep> insertions")
    else:
        print("⚠ Some differences exist beyond <sep> insertions")
        # Show first difference
        for i, (m, d) in enumerate(zip(flat_token_sentence_chunks, flat_token_sentence_chunks)):
            if m != d:
                print(f"  First diff at index {i}: merged='{m}' vs derived='{d}'")
                break
        assert "Difference detected"

    return token_chunks


def chunk_html_by_paragraph(html_content, min_tokens=500):
    """
    Paragraph-based chunker. Uses BeautifulSoup to extract leaf block elements,
    preserving their HTML, then tokenizes + cleans each paragraph, and merges
    consecutive paragraphs until min_tokens is reached.

    Returns: List[List[token]] — same format as the sentence method.
    """
    from bs4 import BeautifulSoup

    body_content = extract_body(html_content)

    # ------------------------------------------------------------------ #
    # 1. Extract leaf block elements (same logic as batch_paragraphs)     #
    # ------------------------------------------------------------------ #
    soup = BeautifulSoup(body_content, "html.parser")

    def get_leaf_blocks(tag):
        leaf_blocks = []
        for child in tag.find_all(
            ["p", "li", "blockquote", "pre", "h1", "h2", "h3", "h4", "h5", "h6"]
        ):
            # Only keep blocks that don't contain other block-level elements
            if not child.find(["p", "li", "blockquote", "pre"]):
                leaf_blocks.append(child)
        return leaf_blocks

    leaf_blocks = get_leaf_blocks(soup)

    # ------------------------------------------------------------------ #
    # 2. Tokenize + clean each paragraph, keeping original HTML           #
    # ------------------------------------------------------------------ #
    paragraph_token_lists = []  # List[List[token]]

    for block in leaf_blocks:
        # str(block) preserves the full HTML of the element (tags included)
        block_html = str(block)

        raw_tokens = tokenize(block_html)
        cleaned_tokens = clean_tokens(
            html_tokens=raw_tokens,
            normalize=True,
            keep_manual_label=True,
            keep_bookmarks=True,
        )

        if cleaned_tokens:
            paragraph_token_lists.append(cleaned_tokens)

    # ------------------------------------------------------------------ #
    # 3. Merge consecutive paragraphs until min_tokens is reached         #
    # ------------------------------------------------------------------ #
    token_chunks = []
    current_chunk = []

    for para_tokens in paragraph_token_lists:
        if not current_chunk:
            # Always start a new chunk with the current paragraph
            current_chunk = list(para_tokens)
        elif len(current_chunk) >= min_tokens:
            # Current chunk is already big enough — flush and start fresh
            token_chunks.append(current_chunk)
            current_chunk = list(para_tokens)
        else:
            # Current chunk is still too small — keep accumulating
            current_chunk.extend(para_tokens)

    # Flush the last chunk
    if current_chunk:
        token_chunks.append(current_chunk)

    print(f"Paragraph chunks: {len(token_chunks)}")
    print(f"Chunk sizes (tokens): {[len(c) for c in token_chunks]}")

    return token_chunks

In [3]:
def main_few_shot_selection(filename, label_config, n_few_shot=30, fs_json_path=None, random_seed=None):
    
    import random
    use_random_mode = random_seed is not None
    
    if use_random_mode:
        random.seed(random_seed)
        if fs_json_path is None:
            fs_json_path = fr"{project_root}\few_shot_selection_tool\second_selected\examples_selected_45_with_sources_fixed_spacing_manual_label.json"
    else:
        if fs_json_path is None:
            fs_json_path = fr"{project_root}\few_shot_selection_tool\second_selected\combined_v3_with_sources_manual_label.json"
    
    # Load the few-shot examples JSON
    with open(fs_json_path, 'r', encoding='utf-8') as file:
        fs_data = json.load(file)
    print(f"   ✓ Loaded {len(fs_data)} examples from: {fs_json_path}")
    print(f"   ✓ Selection mode: {'random' if use_random_mode else 'manual'}")
    
    # Get all unique source files and print them
    source_files = sorted(list(set([item.get('source_file', 'unknown') for item in fs_data])))
    print(f"\n   Source files in few-shot collection:")
    for sf in source_files:
        count = sum(1 for item in fs_data if item.get('source_file') == sf)
        print(f"      - {sf} ({count} examples)")
    
    # Apply filters:
    # 1. Mask filter: Exclude examples from the same document being annotated (avoid data leakage)
    # 2. Selection filter: Either random selection or manual "selected" flag
    current_doc_base = filename
    
    filtered_examples = []
    excluded_same_doc = 0
    excluded_not_selected = 0
    
    for item in fs_data:
        source_file = item.get('source_file', '')
        
        # Filter 1: Check if the current document name appears in the source file (MASK FILTER)
        if current_doc_base in source_file:
            excluded_same_doc += 1
            continue

        # Filter 2: Apply selection filter based on mode
        if use_random_mode:
            # In random selection mode, randomly select from the whole pool (after masking)
            # Keep item if random number is <= probability (n_few_shot / len(fs_data))
            if random.random() > (n_few_shot / len(fs_data)):
                excluded_not_selected += 1
                continue
        else:
            # In manual selection mode, only keep examples where "selected" == true
            if not item.get('selected', False):
                excluded_not_selected += 1
                continue
        
        # Extract input/output from the example
        if 'example' in item and 'input' in item['example'] and 'output' in item['example']:
            filtered_examples.append({
                'input': item['example']['input'],
                'output': item['example']['output'],
                'source_file': source_file
            })
    
    #print(f"\n   ✓ Filtering results:")
    #print(f"      - Excluded (same document): {excluded_same_doc}")
    #print(f"      - Excluded (not selected): {excluded_not_selected}")
    #print(f"      - Retained: {len(filtered_examples)}")
    
    # Show source files of retained examples
    retained_sources = {}
    for ex in filtered_examples:
        sf = ex['source_file']
        retained_sources[sf] = retained_sources.get(sf, 0) + 1
    
    print(f"\n   ✓ Retained examples come from:")
    for sf, count in sorted(retained_sources.items()):
        print(f"      - {sf}: {count} examples")
    
    # Select the required number of examples
    if len(filtered_examples) > n_few_shot:
        selected_examples_dicts = filtered_examples[:n_few_shot]
    else:
        selected_examples_dicts = filtered_examples
    
    
    
    #print(f"\n   ✓ Simplifying outputs to parent-level extraction...")
    simplified_examples = []
    for ex in selected_examples_dicts:
        simplified_output = decode(prepare_label_tokens(tokenize(ex['output']), label_config=label_config))
        simplified_examples.append((ex['input'], simplified_output))
    
    # Convert to list of tuples (input, output)
    selected_few_shot_examples = simplified_examples
    
    #print(f"   ✓ Selected {len(selected_few_shot_examples)} few-shot examples for processing.")
    
    return selected_few_shot_examples

In [4]:
def main_post_processing(processed_chunks, html_content):
    # Processed_chunks is a list of lists of tokens, we need to flatten it to get a single list of tokens for the whole document
    processed_tokens_flat = flatten_token_chunks(processed_chunks)


    # Read in parallel the original tokens and the processed tokens. Always prefer the original tokens, but if there is an auto_label token in the processed tokens, 
    # we want to keep it and merge it with the original tokens. 
    # This way we can keep the original formatting and structure of the document while adding the auto_labels extracted by the model.
    original_tokens = tokenize(html_content)
    processed_html_content_tokens = merge_tokens_general(
        original_tokens=original_tokens,
        derived_tokens=processed_tokens_flat,
        is_protected_func=lambda tok: is_auto_label_tag(tok) != 0,
        is_opening_protected_func=lambda tok: is_auto_label_tag(tok) == 1,
        is_tag_token_func=lambda tok: is_tag_token(tok),
        log=False
        )

    # check of the final processed_html_content with the original HTML, ignoring the auto_label tags which are not present in the original HTML but only in the processed one.
    comparison_result = compare_html_allow_auto_labels(decode(processed_html_content_tokens), html_content)
    assert comparison_result, "The processed HTML content does not match the original HTML content when ignoring auto_label tags. Please check the merging and post-processing steps for errors."


    
    # This merging process can sometimes create some formatting issues with brackets, we need to correct them to get a valid HTML structure.
    processed_html_content_tokens_corrected = correct_tokens_brackets(processed_html_content_tokens)
    assert check_tokens_brackets(processed_html_content_tokens_corrected), "The brackets in the merged tokens are not balanced. Please check the merging and bracket correction steps for errors."


    # The correction of the brackets can sometimes create some redoundant or useless formatting  with the HTML, we need to clean it to compare it with the original.
    processed_html = decode(processed_html_content_tokens_corrected)
    processed_html_cleaned = clean_html_formatting(processed_html)
    print(f"\nMerged HTML length: {len(processed_html_cleaned)}")

    
    # This step is just to ensure a good visualisation of HTMLLabelizer and to add the necessary attribute to stay consistent with the label scheme
    processed_html_content = add_attributes_to_auto_labels(processed_html_cleaned)

    return processed_html_content

In [5]:
def get_all_html_files_in(folder_path):
    """
    Returns:
        dict[str, str]: {filename_without_extension: html_content}
    """
    files = {}
    for entry in os.listdir(folder_path):
        full_path = os.path.join(folder_path, entry)

        if not os.path.isfile(full_path):
            continue

        name, ext = os.path.splitext(entry)
        if ext.lower() not in {".html", ".htm"}:
            continue

        try:
            with open(full_path, "r", encoding="utf-8") as f:
                files[name] = f.read()
        except UnicodeDecodeError:
            with open(full_path, "r", encoding="latin-1") as f:
                files[name] = f.read()

    return files

In [10]:
def get_hyperparameters():
    # ---------- Define Hyperparameters ----------
    min_tokens = 500
    fs_min_tokens = 100
    fs_mode = "pattern"  # "random" or "selected" "pattern"
    model_name = "gpt-5.2"

    n_few_shot = 15  # Number of few-shot examples to use

    prompt_version = "2"
    cot = False
    if cot :
        prompt_path = fr"{project_root}\llm_based_annotation\utils_extraction\prompts\simplified_parent_extraction_cot v{prompt_version}.txt"
    else :
        prompt_path = fr"{project_root}\llm_based_annotation\utils_extraction\prompts\simplified_parent_extraction v{prompt_version}.txt"


    # Define label_config 
    label_config = {
        "keep_attributes": ["labelname"],  # extraction only, no disambiguation
        "switch_type": True,  # manual_label -> auto_label
        "use_simplified": True,  # <auto_label labelname="title"> -> <title>
        "keep_labels": ["decision", "legislation", "secondary sources"]
    }

    return min_tokens, fs_min_tokens, fs_mode, model_name, n_few_shot, prompt_version, prompt_path, cot, label_config


In [11]:
min_tokens, fs_min_tokens, fs_mode, model_name, n_few_shot, prompt_version, prompt_path, cot, label_config = get_hyperparameters()

source_dir = fr"{project_root}\data\Document_Échantillon_Initial\ronde_3\plain_html_arbre_balise"
files = get_all_html_files_in(source_dir)
print(f"✓ Loaded {len(files)} files from: {source_dir}")
#output_dir = fr"{project_root}\data\Documents_Annotés\llm\TEST_PARAGRAPH_CHUNKER_p{prompt_version}_c{min_tokens}_fs{fs_mode}-{n_few_shot}_m{model_name}"

#os.makedirs(output_dir, exist_ok=True)

# Build a set of already processed filenames (case-insensitive), based on *_v1.0.html
#existing_processed = set()
#for entry in os.listdir(output_dir):
#    if not entry.lower().endswith(".html"):
#        continue
#    stem, _ = os.path.splitext(entry)
#    if stem.lower().endswith("_v1.0"):
#        original_name = stem[:-5]  # remove "_v1.0"
#        existing_processed.add(original_name.lower())

✓ Loaded 4 files from: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Document_Échantillon_Initial\ronde_3\plain_html_arbre_balise


In [12]:
filename = "2005QCCA437"
html_content = files[filename]
#if filename.lower() in files:
#    print(f"   ✓ Skipping {filename}.html (already processed)")


token_chunks_par = main_chunk_html(html_content, min_tokens=min_tokens, method="paragraph")




Paragraph chunks: 24
Chunk sizes (tokens): [505, 672, 568, 569, 780, 528, 579, 899, 588, 547, 792, 537, 500, 530, 635, 718, 684, 557, 704, 656, 545, 529, 515, 211]


In [13]:
if fs_mode == "random":
            selected_few_shot_examples = main_few_shot_selection(filename=filename, n_few_shot=n_few_shot, 
                                                                 #fs_json_path=fr"{project_root}\few_shot_selection_tool\second_selected\examples_selected_45_clean_with_sources_fixed_spacing_manual_label.json",
                                                                 random_seed=42,
                                                                 label_config=label_config)  # Set a random seed for reproducibility if using random selection mode 
elif fs_mode == "selected":
    selected_few_shot_examples = main_few_shot_selection(filename=filename, n_few_shot=n_few_shot, 
                                                            fs_json_path=fr"{project_root}\few_shot_selection_tool\second_selected\examples_selected_45_clean_with_sources_fixed_spacing_manual_label.json",
                                                            random_seed=None,
                                                            label_config=label_config)  # Set to None for manual selection mode
elif fs_mode == "pattern":
    selected_few_shot_examples = main_few_shot_selection(filename=filename, n_few_shot=n_few_shot, 
                                                            fs_json_path=fr"{project_root}\few_shot_selection_tool\greedy_set_coverage_rejected_corrected.json",
                                                            random_seed=None,
                                                            label_config=label_config)  # Set to None for manual selection mode


   ✓ Loaded 30 examples from: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\greedy_set_coverage_rejected_corrected.json
   ✓ Selection mode: manual

   Source files in few-shot collection:
      - few_shot_examples_1989CanLII1415ONCA_annotated_GL_tech.json (2 examples)
      - few_shot_examples_1997CanLII16226_ONCA_annotated_EG_tech.json (4 examples)
      - few_shot_examples_2001CanLII21117QCTDP_annotated_GL_tech.json (8 examples)
      - few_shot_examples_2019SCC65_annotated_EG_revRL.json (10 examples)
      - few_shot_examples_2021QCCA1675_annotated_EG_tech.json (2 examples)
      - few_shot_examples_2024NBKB203_annotated_VP.json (4 examples)

   ✓ Retained examples come from:
      - few_shot_examples_1989CanLII1415ONCA_annotated_GL_tech.json: 2 examples
      - few_shot_examples_1997CanLII16226_ONCA_annotated_EG_tech.json: 4 examples
      - few_shot_examples_2001CanLII21117QCTDP_annotated_GL_tech.json: 8 examples
      - few_shot_example

In [15]:
text_test = decode(token_chunks_par[0])

In [20]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'analysis/pattern_analysis'))
sys.path.append(str(Path.cwd().parent / 'few_shot_selection_tool'))
from normalization import normalize_fragment, normalize_decision_citation, normalize_legislation_citation, normalize_secondary_source, normalize_decision_title



In [ ]:
text_test_tokenized_fragment = normalize_fragment(text_test)
text_test_tokenized_decision_citation = normalize_decision_citation(text_test)
text_test_tokenized_legislation_citation = normalize_legislation_citation(text_test)
text_test_tokenized_secondary_source = normalize_secondary_source(text_test)
text_test_tokenized_decision_title = normalize_decision_title(text_test)



In [14]:
import json
with open(fr"{project_root}\few_shot_selection_tool\pattern_dicts.json", 'r', encoding='utf-8') as f:
    pattern_dicts = json.load(f)


In [22]:
print(pattern_dicts["decision_fragment"])

[[49.3, ['PARA', 'NUM']], [16.36, ['P', 'NUM']], [12.65, ['paraSECTION', 'NUM-NUM']], [6.96, ['NUM']], [4.99, ['PP', 'NUM-NUM']], [4.06, ['paraSECTION', 'NUM']], [1.86, ['NUM-NUM']], [0.7, ['PP', 'NUM']], [0.58, ['paraSECTION', 'NUM‑NUM']], [0.46, ['paraSECTION', 'NUM', 'ETSEQ']], [0.46, ['PARA', 'NUM-NUM']], [0.35, ['paraSECTION', 'NUM', 'AND', 'NUM']], [0.23, ['PARANUM']], [0.23, ['NUM', 'AND', 'NUM']], [0.23, ['NUM‑NUM']], [0.23, ['para', 'NUM']], [0.12, ['FOOTNOTE', 'NUM']], [0.12, ['paraSECTION', 'NUM', ',', 'NUM', ',', 'NUM']], [0.12, ['NUM', '-', 'NUM']]]


In [15]:
def _contains_pattern(token_sequence: list[str], pattern: list[str]) -> bool:
    """
    Sliding-window search: returns True if `pattern` appears as a
    consecutive subsequence anywhere inside `token_sequence`.
    """
    n, p = len(token_sequence), len(pattern)
    if p == 0 or p > n:
        return False
    for i in range(n - p + 1):
        if token_sequence[i : i + p] == pattern:
            return True
    return False

def find_matching_patterns(
    token_sequence: list[str],
    pattern_list: list[list],
) -> list[list]:
    """
    Returns all [score, pattern] entries from `pattern_list` whose
    pattern is found as a consecutive subsequence in `token_sequence`.
    """
    return [
        entry                          # [score, pattern]
        for entry in pattern_list
        if _contains_pattern(token_sequence, entry[1])
    ]
 
 
# ── 4. Main metadata builder ──────────────────────────────────────────────────
 
def extract_pattern_metadata(text: str) -> dict[str, list[list]]:
    """
    Tokenizes `text` with every normalization function, checks each
    token sequence against the relevant pattern dictionaries, and
    returns a metadata dict containing only the categories where at
    least one pattern matched.
 
    Parameters
    ----------
    text : str
        A single text chunk (e.g. the first chunk of a document).
 
    Returns
    -------
    dict
        Keys are category names (e.g. "decision_citation").
        Values are lists of [score, pattern] for every matched pattern.
        Categories with zero matches are omitted.
 
    Example output
    --------------
    {
        "decision_fragment":    [[49.3, ["PARA", "NUM"]]],
        "legislation_fragment": [[49.3, ["PARA", "NUM"]], [16.36, ["P", "NUM"]]],
        "decision_citation":    [[49.3, ["PARA", "NUM"]]],
    }
    """
 
    # ── 4a. Tokenize once per normalization function ──────────────────────────
    tok_fragment          = normalize_fragment(text)
    tok_decision_citation = normalize_decision_citation(text)
    tok_legislation_citation = normalize_legislation_citation(text)
    tok_secondary_source  = normalize_secondary_source(text)
    tok_decision_title    = normalize_decision_title(text)
 
    # ── 4b. Map each (token_sequence, category) pair ─────────────────────────
    # normalize_fragment feeds THREE categories (same tokens, different dicts)
    checks: list[tuple[list[str], str]] = [
        (tok_fragment,             "decision_fragment"),
        (tok_fragment,             "legislation_fragment"),
        (tok_fragment,             "sec_sources_fragment"),
        (tok_decision_citation,    "decision_citation"),
        (tok_legislation_citation, "legislation_citation"),
        (tok_secondary_source,     "sec_sources_source"),
        (tok_decision_title,       "decision_title"),
    ]
 
    # ── 4c. Run matching and collect results ──────────────────────────────────
    metadata: dict[str, list[list]] = {}
 
    for token_sequence, category in checks:
        matches = find_matching_patterns(token_sequence, pattern_dicts[category])
        if matches:
            metadata[category] = matches
 
    return metadata

In [27]:
result = extract_pattern_metadata(text_test)
 
print("=== Pattern Metadata ===")
for category, matched_patterns in result.items():
    print(f"\n{category}:")
    for score, pattern in matched_patterns:
        print(f"  score={score:>6.2f}  pattern={pattern}")

=== Pattern Metadata ===

decision_fragment:
  score=  6.96  pattern=['NUM']
  score=  0.23  pattern=['NUM', 'AND', 'NUM']

legislation_fragment:
  score=  6.53  pattern=['NUM']
  score=  4.79  pattern=['ARTICLE', 'NUM']

sec_sources_fragment:
  score=  6.11  pattern=['NUM']

decision_citation:
  score= 26.55  pattern=['YEAR', 'TRIBUNAL', 'NUM']

legislation_citation:
  score=  0.95  pattern=['NUM']

sec_sources_source:
  score=  8.43  pattern=['EDITION']
  score=  3.61  pattern=['YEAR']

decision_title:
  score= 46.69  pattern=['PARTY']
  score= 28.89  pattern=['PARTY', 'V', 'PARTY']
  score= 12.36  pattern=['PARTY', 'V', 'PARTY', '(', 'PARTY', ')']
  score=  0.82  pattern=['PARTY', '(', 'PARTY', ')']
  score=  0.67  pattern=['PARTY', 'V', 'PARTY', '(', 'PARTY', ')', 'PARTY']
  score=  0.52  pattern=['PARTY', '(', 'PARTY', ')', 'PARTY']
  score=  0.07  pattern=['PARTY', '(', 'PARTY']


In [17]:
"""
few_shot_selection.py
─────────────────────
Combines:
  • n_general  fixed/random examples (same logic as main_few_shot_selection)
  • n_dynamic  greedy-coverage examples driven by the chunk's pattern metadata
"""

import json
import random
from typing import Optional

# ── adjust to your project ────────────────────────────────────────────────────
# from your_module import decode, prepare_label_tokens, tokenize, project_root
# ─────────────────────────────────────────────────────────────────────────────


# ═════════════════════════════════════════════════════════════════════════════
# HELPERS
# ═════════════════════════════════════════════════════════════════════════════

def _load_json(path: str) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _simplify_output(raw_output, label_config):
    """Re-uses your existing decode/prepare_label_tokens/tokenize pipeline."""
    return decode(prepare_label_tokens(tokenize(raw_output), label_config=label_config))


def _coverage_set(label_pattern: dict) -> set[tuple]:
    """
    Converts a label_pattern dict from a JSON example into a set of
    (category, tuple(pattern)) pairs for O(1) lookup.

    label_pattern = {
        "decision_fragment": [["PARA", "NUM"], ...],
        "legislation_fragment": [],
        ...
    }
    → {("decision_fragment", ("PARA", "NUM")), ...}
    """
    covered = set()
    for category, patterns in label_pattern.items():
        for pat in patterns:
            covered.add((category, tuple(pat)))
    return covered


def _metadata_to_needs(metadata: dict) -> set[tuple]:
    """
    Converts extract_pattern_metadata() output into the same
    (category, tuple(pattern)) format.

    metadata = {
        "decision_fragment": [[6.96, ["NUM"]], [0.23, ["NUM","AND","NUM"]]],
        ...
    }
    → {("decision_fragment", ("NUM",)), ("decision_fragment", ("NUM","AND","NUM")), ...}
    """
    needs = set()
    for category, scored_patterns in metadata.items():
        for _score, pat in scored_patterns:
            needs.add((category, tuple(pat)))
    return needs


def _mask_same_doc(items: list[dict], filename: str) -> list[dict]:
    return [item for item in items if filename not in item.get("source_file", "")]


def _extract_example(item: dict, label_config) -> dict:
    """Returns a ready-to-use dict with input/output/source_file/coverage."""
    ex = item.get("example", {})
    return {
        "input":       ex.get("input", ""),
        "output":      ex.get("output", ""),
        "source_file": item.get("source_file", ""),
        "coverage":    _coverage_set(item.get("label_pattern", {})),
    }


# ═════════════════════════════════════════════════════════════════════════════
# GENERAL SELECTION  (mirrors main_few_shot_selection)
# ═════════════════════════════════════════════════════════════════════════════

def _select_general(
    fs_data:       list[dict],
    filename:      str,
    n_general:     int,
    label_config,
    use_random:    bool,
    rng:           random.Random,
) -> tuple[list[tuple], set[tuple]]:
    """
    Returns:
      selected_examples  – list of (input, simplified_output)
      covered_patterns   – set of (category, pattern_tuple) already covered
    """
    masked = _mask_same_doc(fs_data, filename)

    if use_random:
        prob = n_general / max(len(masked), 1)
        pool = [item for item in masked if rng.random() <= prob]
    else:
        pool = [item for item in masked if item.get("selected", False)]

    pool = pool[:n_general]

    examples   = []
    covered    = set()
    for item in pool:
        ex = _extract_example(item, label_config)
        simplified = _simplify_output(ex["output"], label_config)
        examples.append((ex["input"], simplified))
        covered |= ex["coverage"]

    return examples, covered


# ═════════════════════════════════════════════════════════════════════════════
# DYNAMIC SELECTION  (greedy set-cover)
# ═════════════════════════════════════════════════════════════════════════════

def _select_dynamic(
    dyn_data:         list[dict],
    filename:         str,
    n_dynamic:        int,
    label_config,
    already_covered:  set[tuple],
    chunk_needs:      set[tuple],
    rng:              random.Random,
) -> list[tuple]:
    """
    Greedy set-cover over the patterns still needed after general selection.
    At each step picks the candidate that covers the most remaining patterns,
    with random tie-breaking.  Stops when n_dynamic examples are chosen or
    all patterns are covered.
    """
    remaining = chunk_needs - already_covered
    if not remaining or n_dynamic == 0:
        return []

    # Build candidate pool (mask same doc, must cover ≥1 needed pattern)
    candidates = []
    for item in _mask_same_doc(dyn_data, filename):
        ex       = _extract_example(item, label_config)
        relevant = ex["coverage"] & remaining          # patterns it can cover
        if relevant:
            candidates.append({
                "input":    ex["input"],
                "output":   ex["output"],
                "coverage": ex["coverage"],
                "relevant": relevant,
            })

    selected   = []
    covered    = set(already_covered)                  # local copy

    while candidates and len(selected) < n_dynamic and remaining:

        # Score each candidate by how many *still* uncovered patterns it adds
        best_gain = max(len(c["coverage"] & remaining) for c in candidates)

        # All candidates tied at best_gain → random tie-break
        top = [c for c in candidates if len(c["coverage"] & remaining) == best_gain]
        chosen = rng.choice(top)

        simplified = _simplify_output(chosen["output"], label_config)
        selected.append((chosen["input"], simplified))

        # Update remaining
        covered   |= chosen["coverage"]
        remaining  = chunk_needs - covered

        # Remove chosen + any candidate now fully redundant (gain == 0)
        candidates = [
            c for c in candidates
            if c is not chosen and len(c["coverage"] & remaining) > 0
        ]

    return selected


# ═════════════════════════════════════════════════════════════════════════════
# PUBLIC API
# ═════════════════════════════════════════════════════════════════════════════

def select_few_shot_for_chunk(
    chunk_metadata:   dict,
    filename:         str,
    label_config,
    n_general:        int                = 20,
    n_dynamic:        int                = 10,
    general_json_path: Optional[str]    = None,
    dynamic_json_path: Optional[str]    = None,
    random_seed:      Optional[int]     = None,
) -> list[tuple]:
    """
    Build a few-shot example list for a single chunk.

    Parameters
    ----------
    chunk_metadata   : output of extract_pattern_metadata(text_chunk)
    filename         : base name of the document being annotated (mask filter)
    label_config     : passed through to decode/prepare_label_tokens/tokenize
    n_general        : max examples from the general/fixed pool
    n_dynamic        : max additional examples chosen by greedy pattern coverage
    general_json_path: path to the general few-shot JSON
                       (defaults to combined_v3_with_sources_manual_label.json)
    dynamic_json_path: path to the dynamic/coverage JSON
                       (defaults to greedy_set_coverage_rejected_corrected.json)
    random_seed      : if set, enables random general selection mode

    Returns
    -------
    list of (input_text, simplified_output) tuples,
    length ≤ n_general + n_dynamic.
    """
    use_random = random_seed is not None
    rng        = random.Random(random_seed)

    # ── resolve paths ─────────────────────────────────────────────────────────
    if general_json_path is None:
        if use_random:
            general_json_path = (
                fr"{project_root}\few_shot_selection_tool\second_selected"
                r"\examples_selected_45_with_sources_fixed_spacing_manual_label.json"
            )
        else:
            general_json_path = (
                fr"{project_root}\few_shot_selection_tool\second_selected"
                r"\combined_v3_with_sources_manual_label.json"
            )

    if dynamic_json_path is None:
        dynamic_json_path = (
            fr"{project_root}\few_shot_selection_tool"
            r"\greedy_set_coverage_rejected_corrected.json"
        )

    # ── load data ─────────────────────────────────────────────────────────────
    general_data = _load_json(general_json_path)
    dynamic_data = _load_json(dynamic_json_path)

    print(f"   ✓ General pool : {len(general_data)} examples from {general_json_path}")
    print(f"   ✓ Dynamic pool : {len(dynamic_data)} examples from {dynamic_json_path}")
    print(f"   ✓ Mode         : {'random' if use_random else 'manual'}")

    # ── convert chunk metadata to needed (category, pattern) pairs ────────────
    chunk_needs = _metadata_to_needs(chunk_metadata)
    print(f"   ✓ Patterns needed by chunk : {len(chunk_needs)}")

    # ── step 1: general selection ─────────────────────────────────────────────
    general_examples, already_covered = _select_general(
        general_data, filename, n_general, label_config, use_random, rng
    )
    print(f"   ✓ General examples selected : {len(general_examples)}")
    print(f"   ✓ Patterns already covered  : {len(already_covered & chunk_needs)}"
          f" / {len(chunk_needs)}")

    # ── step 2: dynamic greedy selection ──────────────────────────────────────
    dynamic_examples = _select_dynamic(
        dynamic_data, filename, n_dynamic, label_config,
        already_covered, chunk_needs, rng
    )
    print(f"   ✓ Dynamic examples selected : {len(dynamic_examples)}")

    # ── combine & return ──────────────────────────────────────────────────────
    all_examples = general_examples + dynamic_examples
    print(f"   ✓ Total few-shot examples   : {len(all_examples)}")
    return all_examples


# ═════════════════════════════════════════════════════════════════════════════
# BATCH HELPER  (iterate over all chunks)
# ═════════════════════════════════════════════════════════════════════════════

def select_few_shot_for_all_chunks(
    token_chunks_par: list,
    filename:         str,
    label_config,
    n_general:        int             = 20,
    n_dynamic:        int             = 10,
    general_json_path: Optional[str] = None,
    dynamic_json_path: Optional[str] = None,
    random_seed:      Optional[int]  = None,
) -> list[list[tuple]]:
    """
    Runs select_few_shot_for_chunk() for every chunk.

    Parameters
    ----------
    token_chunks_par : list of token sequences (your existing variable)
    All other params  : forwarded to select_few_shot_for_chunk()

    Returns
    -------
    List (one entry per chunk) of few-shot example lists.
    """
    all_chunk_few_shots = []

    for i, chunk in enumerate(token_chunks_par):
        text_chunk = decode(chunk)
        metadata   = extract_pattern_metadata(text_chunk)

        print(f"\n── Chunk {i+1}/{len(token_chunks_par)} ──────────────────────────")
        few_shots = select_few_shot_for_chunk(
            chunk_metadata    = metadata,
            filename          = filename,
            label_config      = label_config,
            n_general         = n_general,
            n_dynamic         = n_dynamic,
            general_json_path = general_json_path,
            dynamic_json_path = dynamic_json_path,
            random_seed       = random_seed,
        )
        all_chunk_few_shots.append(few_shots)

    return all_chunk_few_shots




In [25]:
from utils_extraction import decode

selected = select_few_shot_for_all_chunks(
    token_chunks_par = token_chunks_par,
    filename = filename,
    label_config = label_config,
    n_general = 5,
    n_dynamic = 25,
    general_json_path = fr"{project_root}\few_shot_selection_tool\greedy_set_coverage_rejected_corrected.json",
    dynamic_json_path = fr"{project_root}\few_shot_selection_tool\second_selected\examples_selected_45_with_sources_fixed_spacing_manual_label_with_patterns2.json",
    random_seed = None,
)
    


── Chunk 1/24 ──────────────────────────
   ✓ General pool : 30 examples from C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\greedy_set_coverage_rejected_corrected.json
   ✓ Dynamic pool : 787 examples from C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\second_selected\examples_selected_45_with_sources_fixed_spacing_manual_label_with_patterns2.json
   ✓ Mode         : manual
   ✓ Patterns needed by chunk : 16
   ✓ General examples selected : 5
   ✓ Patterns already covered  : 3 / 16
   ✓ Dynamic examples selected : 1
   ✓ Total few-shot examples   : 6

── Chunk 2/24 ──────────────────────────
   ✓ General pool : 30 examples from C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\greedy_set_coverage_rejected_corrected.json
   ✓ Dynamic pool : 787 examples from C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\second_selected\ex

In [29]:
print(selected[0][0][1])

 even <decision>Housen</decision>, which, as this Court acknowledged, was initially applied by appeal courts with “varying degrees of enthusiasm” (<decision>H.L. v. Canada (Attorney General), [2005] 1 S.C.R. 401, at para. 76</decision>; see also <secondary sources>Paul M. Perell, “The Standard of Appellate Review and The Ironies of Housen v. Nikolaisen” (2004), 28 Adv. Q. 40, at p. 53</secondary sources>; <secondary sources>Mike Madden, “Conquering the Common Law Hydra: A Probably Correct and Reasonable Overview of Current Standards of Appellate and Judicial Review” (2010), 36 Adv. Q. 269, at pp. 278-79 and 293</secondary sources>; <secondary sources>Paul J. Pape and John J. Adair, “Unreasonable review: The losing party and the palpable and overriding error standard” (2008), 27 Adv. J. 6, at p. 8</secondary sources>; <secondary sources>Geoff R. Hall, “Two Unsettled Questions in the Law of Contractual Interpretation: A Call to the Supreme Court of Canada” (2011), 50 Can. Bus. L.J. 434, 

### Processing

In [16]:
model = GPTAssistant(model_name, temperature=1)



processed_chunks = process_chunks(
    model=model,
    token_chunks=token_chunks_par,
    process_prompt_path=prompt_path,
    label_config=label_config,
    few_shot_examples=selected_few_shot_examples,
    output_dir=output_dir,
    filename=filename,
    cot = cot,
    )

with open(f"{output_dir}\\processed_chunks_{filename}.json", "w") as f:
    json.dump(processed_chunks, f)

   ✓ Processing 24 chunks with LLM...
   ✓ Using 30 few-shot examples


Processing chunks: 100%|██████████| 24/24 [02:26<00:00,  6.09s/it]

   ✓ Processing history saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\TEST_PARAGRAPH_CHUNKER_p2_c500_fsselected-30_mgpt-5.2\history_2005QCCA437.json

   ✓ Processing completed:
      - Total chunks: 24
      - Successful: 24
      - Failed: 0
   ✓ Processed chunks saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\TEST_PARAGRAPH_CHUNKER_p2_c500_fsselected-30_mgpt-5.2\processed_chunks_2005QCCA437.json


In [19]:
processed_html_content = main_post_processing(processed_chunks, html_content)
# ---------- Save processed HTML to file ----------
with open(fr"{output_dir}\{filename}_v1.0.html", 'w', encoding='utf-8') as f:
    f.write(processed_html_content)
print(f"   ✓ Processed HTML saved to: {output_dir}")

   ✓ Flattened 24 chunks into 14580 tokens
   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)

Merged HTML length: 132931
   ✓ Processed HTML saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\TEST_PARAGRAPH_CHUNKER_p2_c500_fsselected-30_mgpt-5.2


In [ ]:


for filename, html_content in files.items() :
    if filename.lower() in existing_processed:
        print(f"   ✓ Skipping {filename}.html (already processed)")
        continue

    
    token_chunks = main_chunk_html(html_content, min_tokens=min_tokens, method="paragraph")

    selected_few_shot_examples = main_few_shot_selection(filename=filename, n_few_shot=n_few_shot)

    
    model = GPTAssistant(model_name, temperature=1)



    processed_chunks = process_chunks(
        model=model,
        token_chunks=token_chunks,
        process_prompt_path=prompt_path,
        label_config=label_config,
        few_shot_examples=selected_few_shot_examples,
        output_dir=output_dir,
        filename=filename,
        cot = cot,
        )
    
    with open(f"{output_dir}\\processed_chunks_{filename}.json", "w") as f:
        json.dump(processed_chunks, f)


    processed_html_content = main_post_processing(processed_chunks, html_content)
    # ---------- Save processed HTML to file ----------
    with open(fr"{output_dir}\{filename}_v1.0.html", 'w', encoding='utf-8') as f:
        f.write(processed_html_content)
    print(f"   ✓ Processed HTML saved to: {output_dir}")

   ✓ Flattened 287 chunks into 14540 tokens
Original tokens: 14476 (no <sep>)
Derived tokens: 14540 (with <sep>)

Result: 14762 tokens
Number of <sep> tags: 286
Initial sentences: 14762, After merging: 14500
Using citation threshold: 25% (combined period + number density)
Gap tolerance: 3 consecutive sentences below threshold to end citation section
Note: Only the FIRST citation section is detected; all subsequent sentences are not citations
   ✓ Flattened 25 chunks into 14476 tokens
✓ Perfect match! Derived was indeed original + <sep> insertions
   ✓ Loaded 270 examples from: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\second_selected\combined_v3_with_sources_manual_label.json

   Source files in few-shot collection:
      - few_shot_examples_1989CanLII1415ONCA_annotated_GL_tech.json (35 examples)
      - few_shot_examples_1997CanLII16226_ONCA_annotated_EG_tech.json (155 examples)
      - few_shot_examples_2019SCC65_annotated_EG_tech_correc

Processing chunks: 100%|██████████| 25/25 [02:15<00:00,  5.41s/it]


   ✓ Processing history saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\TEST_p2_c500_fsselected-30_mgpt-5.2\history_2005QCCA437.json

   ✓ Processing completed:
      - Total chunks: 25
      - Successful: 25
      - Failed: 0
   ✓ Processed chunks saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\TEST_p2_c500_fsselected-30_mgpt-5.2\processed_chunks_2005QCCA437.json
   ✓ Flattened 25 chunks into 14720 tokens
   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)

Merged HTML length: 133462
   ✓ Processed HTML saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\TEST_p2_c500_fsselected-30_mgpt-5.2
   ✓ Flattened 414 chunks into 17039 tokens
Original tokens: 16827 (no <sep>)
Derived tokens: 17039 (with <sep>)

Result: 17240 tokens
Number of <sep> tags: 413
Initial sentences: 17240, After merging: 16856

Processing chunks: 100%|██████████| 30/30 [02:29<00:00,  4.98s/it]


   ✓ Processing history saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\TEST_p2_c500_fsselected-30_mgpt-5.2\history_2016NBOMB12.json

   ✓ Processing completed:
      - Total chunks: 30
      - Successful: 30
      - Failed: 0
   ✓ Processed chunks saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\TEST_p2_c500_fsselected-30_mgpt-5.2\processed_chunks_2016NBOMB12.json
   ✓ Flattened 30 chunks into 17107 tokens
   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)

Merged HTML length: 150978
   ✓ Processed HTML saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\TEST_p2_c500_fsselected-30_mgpt-5.2
